In [1]:
from sklearn.base import BaseEstimator, ClassifierMixin
import numpy as np
from collections import Counter

In [2]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature       
        self.threshold = threshold   
        self.left = left            
        self.right = right           
        self.value = value           
        
    def is_leaf_node(self):
        return self.value is not None

In [9]:
class myDecisionTree(BaseEstimator, ClassifierMixin):
    def __init__(self, min_sample_splits: int = 2, max_depth: int = 100):
        self.min_sample_splits = min_sample_splits
        self.max_depth = max_depth
        self.root = None

    def fit(self, X: np.ndarray, y: np.ndarray):
        self.root = self._grow_tree(X, y)
        return self

    def _grow_tree(self, X: np.ndarray, y: np.ndarray, depth: int = 0):
        n, n_features = X.shape
        n_labels = len(np.unique(y))

        if (depth >= self.max_depth or n_labels == 1 or n < self.min_sample_splits):
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)

        feature_idxs = range(n_features)
        best_feature, best_thresh = self._best_split(X, y, feature_idxs)

        if best_feature is None:
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)

        left_idxs, right_idxs = self._split(X[:, best_feature], best_thresh)
        
        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        
        return Node(best_feature, best_thresh, left, right)

    def _best_split(self, X: np.ndarray, y: np.ndarray, feat_idxs):
        best_gain = -1
        split_idx, split_threshold = None, None

        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)

            for thr in thresholds:
                gain = self._gini_gain(y, X_column, thr)

                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_threshold = thr

        return split_idx, split_threshold

    def _gini_gain(self, y: np.ndarray, X_column: np.ndarray, threshold: float):
        parent_gini = self._gini(y)

        left_idxs, right_idxs = self._split(X_column, threshold)
        
        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return -1 
        
        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        e_l, e_r = self._gini(y[left_idxs]), self._gini(y[right_idxs])
        
        child_gini = (n_l / n) * e_l + (n_r / n) * e_r
        return parent_gini - child_gini

    def _gini(self, y: np.ndarray):
        counts = np.bincount(y)
        probabilities = counts / len(y)
        return 1 - np.sum(probabilities ** 2)

    def _split(self, X_column: np.ndarray, split_thresh: float):
        left_idxs = np.argwhere(X_column <= split_thresh).flatten()
        right_idxs = np.argwhere(X_column > split_thresh).flatten()
        return left_idxs, right_idxs

    def _most_common_label(self, y: np.ndarray):
        counter = Counter(y)
        return counter.most_common(1)[0][0]

    def predict(self, X: np.ndarray):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x: np.ndarray, node: Node):
        if node.is_leaf_node():
            return node.value

        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

In [12]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import time

In [13]:
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [14]:
custom_tree = myDecisionTree(max_depth=3)
sklearn_tree = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42)

In [15]:
start_time = time.time()
custom_tree.fit(X_train, y_train)
custom_time = time.time() - start_time

In [16]:
start_time = time.time()
sklearn_tree.fit(X_train, y_train)
sklearn_time = time.time() - start_time

In [17]:
custom_preds = custom_tree.predict(X_test)
sklearn_preds = sklearn_tree.predict(X_test)

In [18]:
print("\n--- PERFORMANCE COMPARISON ---")
print(f"Custom Tree Accuracy:  {accuracy_score(y_test, custom_preds):.4f}")
print(f"Sklearn Tree Accuracy: {accuracy_score(y_test, sklearn_preds):.4f}")
print(f"\nCustom Tree Time:  {custom_time:.4f} seconds")
print(f"Sklearn Tree Time: {sklearn_time:.4f} seconds")


--- PERFORMANCE COMPARISON ---
Custom Tree Accuracy:  0.9386
Sklearn Tree Accuracy: 0.9474

Custom Tree Time:  1.7376 seconds
Sklearn Tree Time: 0.0070 seconds
